Goal: able to find dependency between the property and the associated numerical value

In [1]:
import torch
import transformers
from transformers import BertTokenizerFast, BertForTokenClassification
from TorchCRF import CRF
from torch.utils.data import DataLoader, Dataset
import numpy as np
# from datasets import load_metric

In [2]:
import datasets

In [3]:
import evaluate

In [4]:
label_map = {
    "O": 0,
    "B-PROPERTY": 1,
    "I-PROPERTY": 2,
    "B-YEAR": 3,
    "I-YEAR": 4,
    "B-TIME": 5,
    "I-TIME": 6,
    "B-VALUE_CUR": 7,
    "I-VALUE_CUR": 8,
    "B-VALUE_RAW": 9,
    "I-VALUE_RAW": 10,
    "B-MULTIPLIER": 11,
    "I-MULTIPLIER": 12
}

In [99]:
train_sentences = [
    "Revenue in 2023 was $5M",
    "EBIT increased to 10 million dollars",
    "Profit margin in Q2 2022 was 15%",
    "Net income for 2021 was 4.5 billion",
    "Operating costs in 2020 were $2.5M",
    "Total expenditure in Q1 2021 was $1.5M",
    "Net profit in 2022 was 5 billion dollars",
    "The company generated 7.5 million in revenue",
    "Total cost in 2019 was 4.2 billion",
    "Investment of 12 million was made in 2023"
]

train_labels = [
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE_CUR"],  
    ["B-PROPERTY", "O", "O", "B-VALUE_RAW", "B-MULTIPLIER", "O"],  
    ["B-PROPERTY", "I-PROPERTY", "O", "B-TIME", "B-YEAR", "O", "B-VALUE_RAW"],  
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE_RAW", "B-MULTIPLIER"],  
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE_CUR"],  
    ["B-PROPERTY", "I-PROPERTY", "O", "B-TIME", "B-YEAR", "O", "B-VALUE_CUR"],
    ["B-PROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE_RAW", "B-MULTIPLIER", "O"],
    ["O", "O", "O", "B-VALUE_RAW", "B-MULTIPLIER", "O", "B-PROPERTY"],
    ["B-PROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE_RAW", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-VALUE_RAW", "B-MULTIPLIER", "O", "O", "O", "B-YEAR"]
]

In [100]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [101]:
# def align_labels(sentence, word_labels):
#     """Aligns word-level labels with BERT tokenized subwords, ensuring correct `I-` propagation."""
#     tokens = tokenizer.tokenize(sentence)  # Tokenized sentence (subwords)
#     words = sentence.split()  # List of full words
#     aligned_labels = []

#     word_idx = 0  # Track word index in original labels
#     current_idx = 0
#     current_label = None  # Track current label
#     inside_entity = False  # Whether we're inside a multi-token entity

#     for token in tokens:
#         if token.startswith("##") or (token in words[word_idx - 1] and not words[word_idx - 1].startswith(token)):  
#             # If it's a subword or part of a split number, inherit "I-"
#             if current_label and current_label != "O":
#                 aligned_labels.append("I-" + current_label.split("-")[-1])
#             else:
#                 aligned_labels.append("O")  # Keep "O" for non-entities
#         else:  # New word or standalone token
#             if word_idx < len(word_labels):  # Ensure index is within range
#                 current_label = word_labels[word_idx]  # Get word's label
#                 aligned_labels.append(current_label)
#                 word_idx += 1  # Move to next word
#                 inside_entity = current_label != "O"  # Track if we are in an entity
#             else:
#                 aligned_labels.append("O")  # Default to "O" if out of range
#                 inside_entity = False  # Reset inside entity

#     return tokens, aligned_labels

In [102]:
def align_labels(sentence, word_labels):
    words = sentence.split()
    tokens = []
    aligned_labels = []

    word_idx = 0  # Track word index in original labels

    for word in words:
        sub_tokens = tokenizer.tokenize(word)
        tokens.extend(sub_tokens)

        # Assign correct labels
        first_label = word_labels[word_idx]
        if first_label.startswith("I-"):  
            first_label = "B-" + first_label[2:]  # Convert first occurrence of "I-" to "B-"

        sub_labels = [first_label] + ["I-" + first_label[2:] if first_label != "O" else "O"] * (len(sub_tokens) - 1)
        aligned_labels.extend(sub_labels)

        word_idx += 1

    return tokens, aligned_labels


In [103]:
test_sentence = "$2.5M"
test_labels = ["B-VALUE_CUR"]  # Word-level label before tokenization

tokens, adjusted_labels = align_labels(test_sentence, test_labels)

print(f"Original Sentence: {test_sentence}")
print(f"Tokenized Output: {tokens}")
print(f"Aligned Labels: {adjusted_labels}")

Original Sentence: $2.5M
Tokenized Output: ['$', '2', '.', '5', '##m']
Aligned Labels: ['B-VALUE_CUR', 'I-VALUE_CUR', 'I-VALUE_CUR', 'I-VALUE_CUR', 'I-VALUE_CUR']


In [104]:
for i, sentence in enumerate(train_sentences):
    tokens, adjusted_labels = align_labels(sentence, train_labels[i])
    print(f"Sentence {i}: {sentence}")
    print(f"Tokenized: {tokens}")
    print(f"Aligned Labels: {adjusted_labels}")
    print("-" * 40)

Sentence 0: Revenue in 2023 was $5M
Tokenized: ['revenue', 'in', '202', '##3', 'was', '$', '5', '##m']
Aligned Labels: ['B-PROPERTY', 'O', 'B-YEAR', 'I-YEAR', 'O', 'B-VALUE_CUR', 'I-VALUE_CUR', 'I-VALUE_CUR']
----------------------------------------
Sentence 1: EBIT increased to 10 million dollars
Tokenized: ['e', '##bit', 'increased', 'to', '10', 'million', 'dollars']
Aligned Labels: ['B-PROPERTY', 'I-PROPERTY', 'O', 'O', 'B-VALUE_RAW', 'B-MULTIPLIER', 'O']
----------------------------------------
Sentence 2: Profit margin in Q2 2022 was 15%
Tokenized: ['profit', 'margin', 'in', 'q', '##2', '202', '##2', 'was', '15', '%']
Aligned Labels: ['B-PROPERTY', 'B-PROPERTY', 'O', 'B-TIME', 'I-TIME', 'B-YEAR', 'I-YEAR', 'O', 'B-VALUE_RAW', 'I-VALUE_RAW']
----------------------------------------
Sentence 3: Net income for 2021 was 4.5 billion
Tokenized: ['net', 'income', 'for', '2021', 'was', '4', '.', '5', 'billion']
Aligned Labels: ['B-PROPERTY', 'B-PROPERTY', 'O', 'B-YEAR', 'O', 'B-VALUE_RAW'

In [105]:
def tokenize_and_align_labels(sentences, labels, max_length=20):
    tokenized_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for i, (sentence, word_labels) in enumerate(zip(sentences, labels)):
        tokens, aligned_labels = align_labels(sentence, word_labels)

        # Add [CLS] and [SEP] tokens
        tokens = ["[CLS]"] + tokens + ["[SEP]"]
        aligned_labels = ["O"] + aligned_labels + ["O"]  # Keep "O" for special tokens

        # Convert tokens to input IDs
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)  # Assign mask for each token

        # Convert aligned labels to numerical values using `label_map`
        label_ids = [label_map.get(lbl, 0) for lbl in aligned_labels]

        # Pad sequences to max_length
        padding_length = max_length - len(input_ids)
        if padding_length > 0:
            input_ids += [0] * padding_length  # Pad input IDs with 0 (BERT's padding)
            attention_mask += [0] * padding_length  # Pad attention mask with 0
            label_ids += [-100] * padding_length  # Pad labels with -100 to ignore them in loss

        # Store the tokenized results
        tokenized_inputs["input_ids"].append(input_ids)
        tokenized_inputs["attention_mask"].append(attention_mask)
        tokenized_inputs["labels"].append(label_ids)

        # Debugging Output
        print(f"\n🔹 Sentence {i}: {sentence}")
        print(f"🔸 Tokenized: {tokens}")
        print(f"🔸 Aligned Labels: {aligned_labels}")
        print(f"🔸 Input IDs: {input_ids}")
        print(f"🔸 Attention Mask: {attention_mask}")
        print(f"🔸 Label IDs: {label_ids}")
        print("-" * 60)

    return tokenized_inputs


In [106]:
train_encodings = tokenize_and_align_labels(train_sentences, train_labels)


🔹 Sentence 0: Revenue in 2023 was $5M
🔸 Tokenized: ['[CLS]', 'revenue', 'in', '202', '##3', 'was', '$', '5', '##m', '[SEP]']
🔸 Aligned Labels: ['O', 'B-PROPERTY', 'O', 'B-YEAR', 'I-YEAR', 'O', 'B-VALUE_CUR', 'I-VALUE_CUR', 'I-VALUE_CUR', 'O']
🔸 Input IDs: [101, 6599, 1999, 16798, 2509, 2001, 1002, 1019, 2213, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
🔸 Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
🔸 Label IDs: [0, 1, 0, 3, 4, 0, 7, 8, 8, 0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
------------------------------------------------------------

🔹 Sentence 1: EBIT increased to 10 million dollars
🔸 Tokenized: ['[CLS]', 'e', '##bit', 'increased', 'to', '10', 'million', 'dollars', '[SEP]']
🔸 Aligned Labels: ['O', 'B-PROPERTY', 'I-PROPERTY', 'O', 'O', 'B-VALUE_RAW', 'B-MULTIPLIER', 'O', 'O']
🔸 Input IDs: [101, 1041, 16313, 3445, 2000, 2184, 2454, 6363, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
🔸 Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 

In [107]:
class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

In [108]:
train_dataset = NERDataset(train_encodings)
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [109]:
class BertCRF(torch.nn.Module):
    def __init__(self, num_labels):
        super(BertCRF, self).__init__()
        self.bert = BertForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, return_dict=False)
        emissions = outputs[0]  # Token classification logits

        if labels is not None:
            # Mask -100 values by replacing them with a valid index (e.g., 0)
            labels = labels.clone()  # Avoid modifying original tensor
            labels[labels == -100] = 0  # Replace -100 with a valid label index

            # Compute CRF loss
            loss = -self.crf(emissions, labels, mask=attention_mask.bool(), reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=attention_mask.bool())


In [110]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertCRF(num_labels=len(label_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [111]:
def train_model(num_epochs=3):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in train_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            loss = model(input_ids, attention_mask, labels)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_dataloader):.4f}")

In [112]:
train_model(num_epochs=10)

Epoch 1 Loss: 27.0017
Epoch 2 Loss: 20.0963
Epoch 3 Loss: 16.4505
Epoch 4 Loss: 11.6893
Epoch 5 Loss: 9.0724
Epoch 6 Loss: 6.6669
Epoch 7 Loss: 4.7833
Epoch 8 Loss: 3.5025
Epoch 9 Loss: 2.5290
Epoch 10 Loss: 2.0132


In [113]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt")

    # ✅ Remove `token_type_ids` (not needed for CRF)
    inputs.pop("token_type_ids", None)

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        predictions = model(**inputs)

    predicted_labels = [list(label_map.keys())[pred] for pred in predictions[0]]
    tokens = tokenizer.tokenize(tokenizer.decode(inputs["input_ids"][0]))

    return list(zip(tokens, predicted_labels))

In [114]:
test_text = "Revenue increased to 20 million in 2024"
print(predict(test_text))

[('[CLS]', 'O'), ('revenue', 'B-PROPERTY'), ('increased', 'O'), ('to', 'O'), ('20', 'B-VALUE_RAW'), ('million', 'B-MULTIPLIER'), ('in', 'O'), ('202', 'B-YEAR'), ('##4', 'I-YEAR'), ('[SEP]', 'O')]


In [142]:
from collections import defaultdict

def check_consistency(sentences):
    """Checks if the same property is consistently linked to the same value in each year."""
    property_values_by_year = defaultdict(lambda: defaultdict(set))  # {year: {property: set(values)}}

    for sentence in sentences:
        predictions = predict(sentence)  # Run model prediction
        current_property = None
        current_year = None
        current_value = []

        triplets = set()  # Store complete (property, year, value) triplets

        for token, label in predictions:
            # Track the year
            if label.startswith("B-YEAR"):
                current_year = token
            elif label.startswith("I-YEAR") and current_year:
                current_year += token.replace("##", "")  # Merge subword years like "202" + "##3" → "2023"

            # Track the property
            if label.startswith("B-PROPERTY"):
                if current_property and current_value:  # Store previous triplet if it exists
                    triplets.add((current_year, current_property, " ".join(current_value)))
                current_property = token
                current_value = []  # Reset value tracking
            elif label.startswith("I-PROPERTY") and current_property:
                current_property += " " + token.replace("##", "")  # Merge subword properties

            # Track the value
            if label.startswith("B-VALUE") or label.startswith("I-VALUE") or label.startswith("B-MULTIPLIER"):
                current_value.append(token.replace("##", ""))  # Ensure subword merging

        # Store final triplet after finishing sentence
        if current_property and current_value:
            triplets.add((current_year, current_property, " ".join(current_value)))

        # Store triplets in the dictionary
        for year, prop, val in triplets:
            if year and prop and val:  # Ensure all parts exist
                property_values_by_year[year][prop].add(val)

        # # Debugging Output
        # print(f"\n🔹 Sentence: {sentence}")
        # print(f"🔸 Extracted Triplets: {dict(property_values_by_year)}")

    # 🔍 Identify inconsistencies
    print(f"🔸 Extracted Triplets: {dict(property_values_by_year)}")
    inconsistencies = {}
    for year, properties in property_values_by_year.items():
        for prop, values in properties.items():
            if len(values) > 1:  # If the same property has multiple values in the same year
                inconsistencies.setdefault(year, {})[prop] = values

    if inconsistencies:
        print("no")
        for year, props in inconsistencies.items():
            for prop, values in props.items():
                print(f"Year {year}: {prop} → {values}")
    else:
        print("pass")

    return property_values_by_year, inconsistencies


In [143]:
test_sentences = [
    "Revenue in 2023 was $10M",
    "Revenue in 2023 reached $10M",
    "The operating cost in 2023 was 10 million dollars",
    "The operating cost in 2023 reached 10 million dollars",
    "Net profit in 2022 amounted to $5M",
    "Net profit in 2022 amounted to $5M",
]

property_values, inconsistencies = check_consistency(test_sentences)


🔸 Extracted Triplets: {'2023': defaultdict(<class 'set'>, {'revenue': {'$ 10 m'}, 'cost': {'10 million'}}), '2022': defaultdict(<class 'set'>, {'profit': {'$ 5 m'}})}
pass


In [144]:
test_sentences = [
    "Net profit in 2023 was $7M",
    "Net profit in 2023 was $5M"
]
property_values, inconsistencies = check_consistency(test_sentences)

🔸 Extracted Triplets: {'2023': defaultdict(<class 'set'>, {'profit': {'$ 5 m', '$ 7 m'}})}
no
Year 2023: profit → {'$ 5 m', '$ 7 m'}
